In [2]:
# for root anchor (notebook moved into subfolder)
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np

from validation_metrics import (
    ValidationGate, 
    VoltammogramFidelityIndex,
    assign_nearest_log_class,
    quick_compare
)

# centralized paths for all project files
import paths

In [ ]:
# run validation gate module on physics informed augmented signals

gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.AUGMENTED_SIGNALS_CSV, run_tiers=[1, 2])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)

print(results['pff_df'].groupby('class_uM')['Wasserstein'].agg(['mean', 'median', 'count', 'max']))
print()
print(results['pff_df'].sort_values('Wasserstein', ascending=False).head(15))

print(results.get('delta_acf_peak', 'not in results dict'))
print([k for k in results.keys() if 'acf' in k.lower()])
print (results['pff_df']['Wasserstein'].describe())


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 0.8669  [Good     ]  ║
║  NEW  P

In [4]:
# --- 1. Load Data ---
target_col = 'concentration'

E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

# Real signals and labels
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
y_real = df_real[target_col].values
X_real = df_real.drop(columns=[target_col]).values

# --- 2. Initialize Gate & Extract Features ---
gate = ValidationGate(E, X_real, y_real)

# ACCESS feat_real
feat_real = gate.feat_real  

# Evaluate synthetic data to get feat_synth
results = gate.evaluate_csv(paths.AUGMENTED_SIGNALS_CSV, target_col=target_col)
feat_synth = results['feat_synth']

df_aug = pd.read_csv(paths.AUGMENTED_SIGNALS_CSV)
y_synth = df_aug[target_col].values


# --- 3. Diagnostics ---

# How many synthetic samples per real class, after binning?
real_classes = np.unique(gate.y_real)
y_synth_binned = assign_nearest_log_class(y_synth, real_classes)
print("--- Synthetic Samples per Class ---")
print(pd.Series(y_synth_binned).value_counts().sort_index())

# Manually reproduce ONE cell -- Ep at 2.50 µM -- to see the raw numbers
r = feat_real[gate.y_real == 2.5]['Ep'].values
s = feat_synth[y_synth_binned == 2.5]['Ep'].values

print("\n--- Ep at 2.50 µM ---")
print('real Ep:', r)
print('synth Ep:', s)
print('real std:', r.std(ddof=1), 'synth std:', s.std(ddof=1))
print('pooled std used in normaliser:', np.std(np.concatenate([r, s]), ddof=1))


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ✅ PASS              ║
║    TSTR l

In [ ]:
# timegan test
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.TIMEGAN_SIGNALS_CSV, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.450  (≥0.90)                ║
║    JSD mean / max                     : 0.1088 / 0.5290           ║
║    MMD²                               : 0.132346                ║
║    SWD mean / max                     : 0.1489 / 0.4816           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.167  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9847  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.20179  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [20]:
# wgangp test
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.WGANGP_SIGNALS_CSV, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.688  (≥0.90)                ║
║    JSD mean / max                     : 0.0593 / 0.1908           ║
║    MMD²                               : 0.063509                ║
║    SWD mean / max                     : 0.1379 / 0.4227           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.667  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9538  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.01733  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [17]:

gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results_sanity = gate.evaluate_csv(paths.REAL_SIGNALS_CSV, run_tiers=[1, 2])

vfi_perfect = VoltammogramFidelityIndex.from_gate_results(results_sanity, verbose=True)
print(f"VFI Sanity Check: {vfi_perfect:.4f}") 


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 1.000  (≥0.90)                ║
║    JSD mean / max                     : 0.0000 / 0.0000           ║
║    MMD²                               : -0.024235                ║
║    SWD mean / max                     : 0.0000 / 0.0000           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 1.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9775  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.00000  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 1.0000  [Excellent]  ║
║  NEW  

In [21]:
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': paths.AUGMENTED_SIGNALS_CSV,
    'TimeGAN': paths.TIMEGAN_SIGNALS_CSV,
    'WGANGP': paths.WGANGP_SIGNALS_CSV
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True
1             TimeGAN        0.3118    0.1300  0.192612    0.2928           0.000       0.9523    0.51084  0.5166      Poor            0.6625       False       False
2              WGANGP        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849      Good            0.7610       False       False


In [24]:
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': paths.AUGMENTED_SIGNALS_CSV,
    'TimeGAN': paths.TIMEGAN_SIGNALS_CSV,
    'WGANGP': paths.WGANGP_SIGNALS_CSV
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2, 3, 4])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True
1             TimeGAN        0.3118    0.1300  0.192612    0.2928           0.000       0.9523    0.51084  0.5166      Poor            0.6625       False       False
2              WGANGP        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849      Good            0.7610       False       False
